# Target Leakage Prevention

**Target Leakage** happens when your model is given access to information during training that it would not realistically have access to in the future when making a live prediction. 

It is the equivalent of giving a student the answer key to a math test while they are studying, and then being shocked when they fail the real test where the answer key is hidden.

There are two main types of target leakage:
1. **Feature Leakage (Time Travel)**: Including a column in your data that is actually created *after* the target event happens.
2. **Train-Test Contamination**: Leaking information from your Testing data into your Training data during preprocessing (like Scaling or Imputing).

Let's set up a Python sandbox to see exactly how these leaks happen and how to stop them.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create a dataset for predicting if a customer will default on a loan
np.random.seed(42)

data = {
    'customer_id': range(1, 101),
    'credit_score': np.random.randint(500, 800, 100),
    'income': np.random.randint(30000, 120000, 100),
    # The Target: Did they default? (1 = Yes, 0 = No)
    'will_default': np.random.choice([0, 1], 100),
}

df = pd.DataFrame(data)

# FEATURE LEAKAGE: Adding a column that only happens AFTER they default!
# If they default, the bank hires a collection agency.
df['sent_to_collections'] = df['will_default'] 

print("✅ Loan Dataset created!")
display(df.head())

✅ Loan Dataset created!


,customer_id,credit_score,income,will_default,sent_to_collections
0,1,602,38110,0,0
1,2,770,109309,1,1
2,3,606,57266,0,0
3,4,571,82992,1,1
4,5,688,112948,0,0


# 1. Feature Leakage (Time Travel)
Look closely at the `sent_to_collections` column. In the real world, a bank wants to predict if a customer will default *before* they give them the loan. But a customer is only sent to collections *after* they have already defaulted. 

If you include this column in your training data, the model will learn: *"If `sent_to_collections` is 1, they will default!"* It achieves 100% accuracy. But tomorrow, when a new customer walks in asking for a loan, the `sent_to_collections` field will be empty. The model will panic and fail.

**How to prevent it:** You must relentlessly interrogate every single feature. Ask yourself: *"Will I have this exact piece of data at the exact moment I need to make the prediction?"* If the answer is no, drop it.

In [2]:
# 🚨 PREVENTING FEATURE LEAKAGE 🚨
# We must drop 'sent_to_collections' before training!
df_clean = df.drop(columns=['sent_to_collections', 'customer_id'])

# Define our inputs (X) and target (y)
X = df_clean.drop(columns=['will_default'])
y = df_clean['will_default']

print("✅ Feature Leakage prevented. We only kept data available at the time of application.")

✅ Feature Leakage prevented. We only kept data available at the time of application.


# 2. Train-Test Contamination (The Wrong Way)
Before we train a model, we must split our data into a **Training Set** (to teach the model) and a **Testing Set** (to test it on unseen data). 

A massive mistake beginners make is scaling or imputing their data *before* splitting it. 

In [3]:
# ❌ THE WRONG WAY (Data Contamination) ❌

# 1. The Data Scientist scales the ENTIRE dataset at once
scaler = StandardScaler()
X_scaled_wrong = scaler.fit_transform(X) # <-- LEAKAGE HAPPENS HERE

# 2. Then they split the data
X_train_bad, X_test_bad, y_train, y_test = train_test_split(X_scaled_wrong, y, test_size=0.2)

print("❌ Contamination occurred!")

❌ Contamination occurred!


**Why is this bad?** `StandardScaler` calculates the Mean of the data. Because you ran it on the entire dataset, the "Testing Data" influenced the Mean. Your Training Data now contains mathematical hints about the Testing Data. Your test is no longer a true "blind" test!

# 3. Train-Test Isolation (The Right Way)
To prevent contamination, you must draw a hard firewall between your Train and Test data. You only `fit()` (learn the math) on the Training data. Then you `transform()` the Test data using those saved rules.

In [4]:
# ✅ THE RIGHT WAY (Strict Isolation) ✅

# 1. Split the data FIRST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Initialize the Scaler
scaler = StandardScaler()

# 3. FIT (learn the mean) ONLY on the Training Data, and transform it.
X_train_scaled = scaler.fit_transform(X_train)

# 4. ONLY TRANSFORM the Testing data using the rules learned from Step 3.
X_test_scaled = scaler.transform(X_test) # Notice: NO .fit() here!

print("✅ Data successfully scaled with strict Train-Test isolation.")

✅ Data successfully scaled with strict Train-Test isolation.


# 4. The Ultimate Solution: Scikit-Learn Pipelines
Manually keeping track of what you `fit` and what you `transform` becomes a nightmare when you have missing values, one-hot encoding, and scaling all happening at once.

To guarantee zero target leakage, professionals use **Scikit-Learn Pipelines**. A Pipeline bundles your preprocessing steps and your machine learning model into one single object. It automatically handles the `fit/transform` logic safely behind the scenes.

In [5]:
from sklearn.pipeline import make_pipeline

# 1. Split the raw, untouched data FIRST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Create the Pipeline
# Step A: Scale the data
# Step B: Train a Logistic Regression model
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

# 3. Train the entire pipeline using ONLY the training data
pipeline.fit(X_train, y_train)

# 4. Predict on the test data
# The pipeline automatically transforms X_test using the scaler rules, 
# and then passes it to the model for prediction!
predictions = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, predictions)
print(f"✅ Pipeline ran perfectly safely! Test Accuracy: {accuracy * 100:.2f}%")

✅ Pipeline ran perfectly safely! Test Accuracy: 60.00%


## Real-World Analogy:
Think of Target Leakage like **Predicting the Weather**:

* **The Goal**: Predict if it will rain tomorrow.
* **Feature Leakage (Time Travel)**: You build a model using humidity, wind speed, and *amount of puddles on the ground*. The model gets 100% accuracy in the lab! But in reality, puddles only form *after* it rains. When you try to predict tomorrow's weather today, there are no puddles yet. Your model crashes.
* **Train-Test Contamination (The Stolen Answer Key)**: You are a professor giving a final exam. You want to see how smart your students (the Model) are. 
    * **The Wrong Way**: You calculate the class average using all the exams, tell the students the average, and *then* lock them in the room to take the test. They now have a hint about what the other scores are.
    * **The Right Way (Pipelines)**: You lock the students in the room. They take the test using only the knowledge in their heads (Training Data). You grade the tests in secret (Testing Data). The true intelligence of the student is revealed.

---